## {Gaussian, Uniform} to 8-Gaussian, guidance with Learned, MC, CEG

In [1]:
import sys
sys.path.append('..')
%load_ext autoreload
%autoreload 2

from functools import partial
from typing import List, Tuple
from guided_flow.backbone.mlp import MLP, ClassConditionalMLP, CondMLP
from guided_flow.backbone.wrapper import ExpEnergyMLPWrapper, GuidedMLPWrapper, MLPWrapper, ClassConditionalMLPWrapper
from guided_flow.config.sampling import GuideFnConfig
from guided_flow.distributions.base import BaseDistribution, get_distribution
from guided_flow.distributions.gaussian import GaussianDistribution
from guided_flow.flow.optimal_transport import OTPlanSampler
from guided_flow.guidance.gradient_guidance import wrap_grad_fn
from guided_flow.utils.misc import deterministic, check_energy_compatibility
from guided_flow.utils.metrics import compute_w2 as w2
import torch
from torchdyn.core import NeuralODE
import numpy as np
from torch.distributions import Normal, Independent
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import os, glob
from tqdm import tqdm
from guided_flow.config.sampling import ODEConfig


from guided_flow.utils.kl_divergence import compute_kl_divergence
MLP_WIDTH = 256
MLP_WIDTH_CCFM = 512
TRAINING_B = 256 # OT CFM training batch size


def get_latest_model(base_dir, prefix, distribution=None, current_scale=None):
    """
    Searches for all .pth files matching the prefix and returns the latest one.
    If a distribution is provided, it filters the models from newest to oldest
    to find the latest one that is compatible with the configured energy function and scale.
    """
    search_pattern = os.path.join(base_dir, f"{prefix}*.pth")
    files = glob.glob(search_pattern)
    
    if not files:
        raise FileNotFoundError(f"Could not find any models matching: {search_pattern}")
    
    # Sorting in descending alphabetical order gets the newest timestamp first
    sorted_files = sorted(files, reverse=True)
    
    if distribution is not None:
        for file in sorted_files:
            try:
                check_energy_compatibility(file, distribution, current_scale)
                print(f"Loaded latest compatible model: {os.path.basename(file)}")
                return file
            except ValueError:
                continue
        fallback_file = sorted_files[0]
        print(f"WARNING: Could not find any models matching prefix '{prefix}' that are compatible "
              f"with energy function '{distribution.energy_fn_name}' and scale {current_scale}. "
              f"Falling back to the latest model: {os.path.basename(fallback_file)}")
        return fallback_file
    
    latest_file = sorted_files[0]
    print(f"Loaded latest model: {os.path.basename(latest_file)}")
    
    return latest_file


def sample_x1_frompeJ(x1_sampler, x1_dist, device, B):
    x1 = None
    while x1 is None or x1.shape[0] < B:
        x1_ = x1_sampler(B).to(device)
        weights = torch.exp(-x1_dist.get_J(x1_))
        acc_prob = weights / weights.max()
        random_numbers = torch.rand(B, device=device)
        x1_ = x1_[random_numbers < acc_prob]
        if x1 is None:
            x1 = x1_
        else:
            x1 = torch.cat([x1, x1_], 0)
    x1 = x1[:B]
    return x1


def compute_w2(trajs, cfgs: List[GuideFnConfig]):
    w2s = []
    for traj, cfg in zip(trajs, cfgs):
        x0_dist = get_distribution(cfg.dist_pair[0])
        x1_dist = get_distribution(cfg.dist_pair[1], energy_fn=cfg.energy_fn, **cfg.energy_kwargs)
        
        x1 = sample_x1_frompeJ(x1_dist.sample, x1_dist, cfg.ode_cfg.device, cfg.ode_cfg.batch_size)
        w2s.append(w2(traj[-1], x1))
    return w2s


def get_mc_guide_fn(x0_dist: BaseDistribution, x1_dist: BaseDistribution, mc_cfg: GuideFnConfig, cfm: str):

    def log_cfm_p_t1(x1, xt, t):
        # xt = t x1 + (1 - t) x0 -> x0 = xt / (1 - t) - t / (1 - t) x1
        x0 = xt / (1 - t + mc_cfg.ep) - (t + mc_cfg.ep) / (1 - t + mc_cfg.ep) * x1 # (B, 2)
        p1t = x0_dist.prob(x0).clamp(1e-8) / (1 - t[0] + mc_cfg.ep) ** 2 # (B,)
        log_p1t = p1t.log()
        # print(log_p1t.mean())
        return log_p1t
        
    def ot_cfm_log_p_tz(x0, x1, xt, t, std=None):
        mean = t * x1 + (1 - t) * x0 # (B, 2)
         # g.t. std: 0. Too small: requires large mc_batch_size; Too large: inaccurate
        base_dist = Normal(loc=mean, scale=std)
        distribution = Independent(base_dist, 1)
        log_p1t = distribution.log_prob(xt) # (B,)
        return log_p1t
        
    def guide_fn(t, x, dx_dt, model, x0=None, x1=None, Jx1=None):
        """
        Args:
            t: Tensor, shape (b, 1)
            x: Tensor, shape (b, dim)
            dx_dt: Tensor, shape (b, dim)
            model: MLP
        """
        # estimate E (e^{-J} / Z - 1) * u
        b = x.shape[0]
        B = mc_cfg.mc_batch_size
        x_ = x.repeat(B, 1) # (MC_B * b, 2)
        t_ = t.repeat(B * b, 1) # (MC_B * b)
        if cfm == 'cfm':
            log_p_t1_x = log_cfm_p_t1(x1, x_, t_) # (MC_B * b) # TODO
            log_p_t_x = log_p_t1_x.reshape(B, b, 1).logsumexp(0) - torch.log(torch.tensor(B, device=x.device)) # (MC_B, B, 1) -> (B, 1)
            log_p_t1_x_times_J_ = (log_p_t1_x + torch.log(Jx1)).reshape(B, b, 1) # (MC_B * b) -> (MC_B, b, 1)            
            logZ = torch.logsumexp(log_p_t1_x_times_J_, 0) - torch.log(torch.tensor(B, device=x.device)) - log_p_t_x # (b, 1)

            Z = torch.exp(logZ)
            u = (x1 - x_) / (1 - t_ + mc_cfg.ep) # (MC_B * b, dim)

            g = (log_p_t1_x.reshape(B, b, 1) - log_p_t_x.unsqueeze(0)).exp() * (Jx1.reshape(B, b, 1) / (Z + 1e-8).unsqueeze(0) - 1) * u.reshape(B, b, 2) # (MC_B, b, dim)

            return g.mean(0)
        
        elif cfm in ['ot_cfm', 'entropic_ot_cfm']:
            log_p_tz_x = ot_cfm_log_p_tz(x0, x1, x_, t_, std=mc_cfg.ot_std) # (MC_B * b)
            log_p_t_x = log_p_tz_x.reshape(B, b, 1).logsumexp(0) - torch.log(torch.tensor(B, device=x.device)) # (MC_B, b, 1) -> (b, 1)
            log_p_tz_x_times_J_ = (log_p_tz_x + torch.log(Jx1)).reshape(B, b, 1) # (MC_B * b) -> (MC_B, b, 1)
            
            logZ = torch.logsumexp(log_p_tz_x_times_J_, 0) - torch.log(torch.tensor(B, device=x.device)) - log_p_t_x # (b, 1)
            
            Z = torch.exp(logZ)
            u = x1 - x0 # (MC_B * b, dim)
            
            g = (log_p_tz_x.reshape(B, b, 1) - log_p_t_x.unsqueeze(0)).exp() * (Jx1.reshape(B, b, 1) / Z - 1) * u.reshape(B, b, 2) # (MC_B, b, dim)
            
            return g.mean(0)

    
    if cfm == 'cfm':
        x1 = x1_dist.sample(mc_cfg.mc_batch_size).to(mc_cfg.ode_cfg.device).unsqueeze(0).repeat(mc_cfg.ode_cfg.batch_size, 1, 1).permute(1, 0, 2).reshape(-1, 2)
        Jx1 = torch.exp(-mc_cfg.scale * x1_dist.get_J(x1))
        return partial(
            guide_fn, 
            x1=x1, 
            Jx1=Jx1
        )
    elif cfm in ['ot_cfm', 'entropic_ot_cfm']:
        x0 = x0_dist.sample(mc_cfg.mc_batch_size) # (MC_B, 2)
        x1 = x1_dist.sample(mc_cfg.mc_batch_size) # (MC_B, 2)
        x0_ = x0.to(mc_cfg.ode_cfg.device).unsqueeze(0).repeat(mc_cfg.ode_cfg.batch_size, 1, 1).permute(1, 0, 2).reshape(-1, 2)
        x1_ = x1.to(mc_cfg.ode_cfg.device).unsqueeze(0).repeat(mc_cfg.ode_cfg.batch_size, 1, 1).permute(1, 0, 2).reshape(-1, 2)
        J_ = torch.exp(-mc_cfg.scale * x1_dist.get_J(x1_)) # (MC_B * b)
        
        return partial(
            guide_fn, 
            x0=x0_,
            x1=x1_,
            Jx1=J_
        )

def get_guide_fn(dist: BaseDistribution, cfg: GuideFnConfig):
    def guide_fn(t, x, dx_dt, model):

        if cfg.guide_type == 'g_cov_A':
            x1_pred = x + dx_dt * (1 - t)
            J = dist.get_J(x1_pred)
            try:
                with torch.enable_grad():
                    x1_pred = x1_pred.requires_grad_(True)
                    J = dist.get_J(x1_pred)
                    grad = -torch.autograd.grad(J.sum(), x1_pred, create_graph=True)[0]
                    return grad
            except Exception as e:
                return torch.zeros_like(x)
        
        elif cfg.guide_type == 'g_cov_G':
            with torch.enable_grad():
                x = x.requires_grad_(True)
                x1_pred = x + model(torch.cat([x, t.repeat(x.shape[0])[:, None]], 1)) * (1 - t)
                J = dist.get_J(x1_pred)
                try:
                    grad = -torch.autograd.grad(J.sum(), x, create_graph=True)[0]
                    return grad
                except Exception as e:
                    return torch.zeros_like(x)
        else:
            raise ValueError(f"Unknown guide function: {cfg.guide_type}")
    # make scale and schedule
    return wrap_grad_fn(cfg.guide_scale, cfg.guide_schedule, guide_fn)

def get_sim_mc_guide_fn(x1_dist: BaseDistribution, cfg: GuideFnConfig):
    def guide_fn(t, x, dx_dt, model):
        """
        Implements guidance following Eq. 12
        Args:
            t: flow time. float
            x: current sample x_t. Tensor, shape (b, dim)
            dx_dt: current predicted VF. Tensor, shape (b, dim)
            model: flow model. MLP
        """
        x1_pred = x + dx_dt * (1 - t) # (B, 2)
        std = cfg.sim_mc_std
        
        x1 = torch.randn_like(x1_pred.unsqueeze(0).repeat(cfg.sim_mc_n, 1, 1)) * std + x1_pred # (cfg.sim_mc_n, B, 2)
        Jx1_ = torch.exp(-cfg.scale * x1_dist.get_J(x1.reshape(-1, 2))).reshape(cfg.sim_mc_n, -1) # (cfg.sim_mc_n, B)
        v = (x1 - x) / (1 - t + cfg.ep)  # Conditional VF v_{t|z} in Eq. 12 (cfg.sim_mc_n, B, 2)
        Z = Jx1_.mean(0) + 1e-8  # Z in Eq. 12 (B,)
        g = (Jx1_ / Z - 1).unsqueeze(2) * v  # g in Eq. 12 (cfg.sim_mc_n, B, 2)
        return g.mean(0)
    return wrap_grad_fn(cfg.guide_scale, cfg.guide_schedule, guide_fn)

def evaluate(x0_sampler, x1_sampler, model, guide_fn, cfg: ODEConfig):
    node = NeuralODE(
        GuidedMLPWrapper(
            model, 
            guide_fn=guide_fn,
            scheduler=lambda t: 1
        ), 
        solver="euler", sensitivity="adjoint")

    with torch.no_grad():
        traj = node.trajectory(
            x0_sampler(cfg.batch_size).to(cfg.device), 
            t_span=torch.linspace(0, cfg.t_end, cfg.num_steps, dtype=torch.float32).to(cfg.device)
        )
    
    return traj


def sample_and_compute_w2(guide_cfgs: List[GuideFnConfig], show_plots: bool = False):
    print("Monte Carlo batch size:", guide_cfgs[0].mc_batch_size)

    trajs = []

    for cfg in guide_cfgs:

        # Initialize samplers, model and guidance model
        x0_dist = get_distribution(cfg.dist_pair[0])
        x1_dist = get_distribution(cfg.dist_pair[1], energy_fn=cfg.energy_fn, **cfg.energy_kwargs)

        x0_sampler = x0_dist.sample
        x1_sampler = x1_dist.sample

        model = MLP(dim=2, w=MLP_WIDTH, time_varying=True).to(cfg.ode_cfg.device)
    
        # Define the base directory where the models for this pair are saved
        log_dir = f'../logs/{cfg.dist_pair[0]}-{cfg.dist_pair[1]}/{cfg.cfm}_{cfg.dist_pair[0]}_{cfg.dist_pair[1]}/'
        
        # Define the prefix for the base model (everything before the timestamp)
        prefix = f'{cfg.cfm}_{cfg.dist_pair[0]}_{cfg.dist_pair[1]}'
        
        # Grab the newest file automatically
        latest_cfm_path = get_latest_model(log_dir, prefix)
        
        # Load the weights
        model.load_state_dict(torch.load(latest_cfm_path, map_location='cpu', weights_only=True))
        if cfg.guide_type == 'mc':
            # sample using flow model
            traj = evaluate(x0_sampler, x1_sampler, model, get_mc_guide_fn(x0_dist, x1_dist, cfg, cfg.cfm), cfg.ode_cfg)
            
        elif cfg.guide_type == 'learned':
            model_G = MLP(dim=2, out_dim=2, w=MLP_WIDTH, time_varying=True).to(cfg.ode_cfg.device)
            gm_prefix = f'guidance_matching_{cfg.gm_type}_scale_{cfg.scale}_{cfg.dist_pair[0]}_{cfg.dist_pair[1]}'
            latest_gm_path = get_latest_model(log_dir, gm_prefix, distribution=x1_dist, current_scale=cfg.scale)
            model_G.load_state_dict(torch.load(latest_gm_path, map_location='cpu', weights_only=True))
            traj = evaluate(x0_sampler, x1_sampler, model, MLPWrapper(model_G, scheduler=lambda t: 1., clamp=0), cfg.ode_cfg)
        
        elif cfg.guide_type == 'ceg':
            model_Z = MLP(dim=2, out_dim=1, w=MLP_WIDTH, time_varying=True, exp_final=False).to(cfg.ode_cfg.device)
            ceg_prefix = f'ceg_scale_{cfg.scale}_{cfg.dist_pair[0]}_{cfg.dist_pair[1]}'
            latest_ceg_path = get_latest_model(log_dir, ceg_prefix, distribution=x1_dist, current_scale=cfg.scale)
            model_Z.load_state_dict(torch.load(latest_ceg_path, map_location='cpu', weights_only=True))
            
            # 2D xy plane. make uniform grid
            XX = torch.linspace(0, 1, 100)
            YY = torch.linspace(0, 1, 100)
            # Use 'xy' indexing so it matches standard cartesian plotting
            XX, YY = torch.meshgrid(XX, YY, indexing='xy') 
            xy = torch.stack([XX.flatten(), YY.flatten()], 1)
            t = torch.zeros(10000, 1) + 0.9
            
            # Calculate the energy landscape
            Z_grid = model_Z(torch.cat([xy, t], 1).to(cfg.ode_cfg.device)).detach().cpu().numpy().reshape(100, 100)
            
            if show_plots:
                # Plot with correct coordinate mapping
                fig, ax = plt.subplots(figsize=(6, 5))
                
                # Use extent=[xmin, xmax, ymin, ymax] to map indices back to 0-1
                img = ax.imshow(Z_grid, extent=[0, 1, 0, 1], origin='lower', cmap='viridis', aspect='auto')
                
                # Add descriptive labels
                ax.set_title(f"CEG Learned Energy Landscape Z(x, t=0.9)\n{cfg.dist_pair[0]} -> {cfg.dist_pair[1]}")
                ax.set_xlabel("X Coordinate")
                ax.set_ylabel("Y Coordinate")
                fig.colorbar(img, ax=ax, label="Learned Scalar Value Z")
                
                plt.show() # Display it cleanly before returning the trajectory
            
            traj = evaluate(x0_sampler, x1_sampler, model, ExpEnergyMLPWrapper(model_Z, scheduler=lambda t: 1, clamp=1), cfg.ode_cfg)
        elif cfg.guide_type in ['g_cov_A', 'g_cov_G']:
            traj = evaluate(x0_sampler, x1_sampler, model, get_guide_fn(x1_dist, cfg), cfg.ode_cfg)

        elif cfg.guide_type == 'g_sim_MC':
            traj = evaluate(x0_sampler, x1_sampler, model, get_sim_mc_guide_fn(x1_dist, cfg), cfg.ode_cfg)
        trajs.append(traj)
    return trajs, None

deterministic(0)


/Users/LS/miniconda3/envs/bio_ml/lib/python3.12/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/LS/miniconda3/envs/bio_ml/lib/python3.12/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <367D4265-B20F-34BD-94EB-4F3EE47C385B> /Users/LS/miniconda3/envs/bio_ml/lib/python3.12/site-packages/torchvision/image.so
  Reason: tried: '/Users/LS/miniconda3/envs/bio_ml/lib/python3.12/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/LS/miniconda3/envs/bio_ml/lib/python3.12/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/LS/miniconda3/envs/bio_ml/lib/python3.12/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/LS/miniconda3/envs/bio_ml/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with

In [2]:
def run_and_plot_comparisons(dist_pairs: List[Tuple], device: str = None, show_energy_plots: bool = False, cfm: str = 'cfm') -> Tuple[plt.Figure, dict]:
    """
    Evaluates CEG, Sim-MC, Guidance A, Guidance G, MC-CFM, GM-CFM, and CC-CFM (if available)
    for a list of source/target distribution name pairs, and plots their trajectories.
    
    Args:
        dist_pairs: A list of tuples of strings, e.g. [('circle', 's_curve'), ('uniform', '8gaussian'), ...]
        device: Optional PyTorch device to use. If None, it will be automatically resolved.
        show_energy_plots: If True, displays CEG energy function landscape plots.
        cfm: The coupling method ('cfm' or 'ot_cfm'). Defaults to 'cfm'.
        
    Returns:
        fig: The plotted Figure object containing rows of comparisons.
        all_results: A dict mapping each pair to evaluated trajectories and configurations.
    """
    if device is None:
        from guided_flow.utils.misc import get_device
        device = get_device()
        
    print(f"Using device: {device}")
    
    guide_cfgs_uncond = []
    guide_cfgs_ceg = []
    guide_cfgs_sim_mc = []
    guide_cfgs_g_cov_a = []
    guide_cfgs_g_cov_g = []
    guide_cfgs_mc = []
    guide_cfgs_gm = []
    
    for pair_info in dist_pairs:
        if len(pair_info) == 2:
            x0, x1 = pair_info
            energy_fn = 'default'
            energy_kwargs = {}
        elif len(pair_info) == 3:
            x0, x1, energy_fn = pair_info
            energy_kwargs = {}
        else:
            x0, x1, energy_fn, energy_kwargs = pair_info
        # Default / tuned hyperparameters matching the notebook's hardcoded values:
        ep = 1e-2
        mc_ep = 1e-2
        t_end = 1.0
        ceg_t_end = 0.95
        mc_t_end = 1.0
        mc_batch_size = 1024
        g_scale = 1.0
        g_schedule = 'linear_decay'
        sim_mc_scale = 1.0
        sim_mc_std = 1.0
        
        if x1 == 's_curve':
            ep = 1e-2
            mc_ep = 5e-2
            mc_t_end = 0.95
            ceg_t_end = 0.95
            mc_batch_size = 10240
            g_scale = 0.2
            g_schedule = 'exp_decay'
            sim_mc_scale = 2.0
            sim_mc_std = 1.0
        elif x1 == '8gaussian':
            ep = 1e-3
            mc_ep = 1e-3
            mc_t_end = 1.0
            ceg_t_end = 0.95
            mc_batch_size = 10240
            g_scale = 0.05  # TODO adjust the guide_scale here 
            g_schedule = 'linear_decay'
            sim_mc_scale = 1.0
            sim_mc_std = 1.0
        elif x1 == 'moon':
            ep = 1e-2
            mc_ep = 1e-2
            mc_t_end = 1.0
            ceg_t_end = 0.95
            mc_batch_size = 1024
            g_scale = 2.0
            g_schedule = 'linear_decay'
            sim_mc_scale = 1.0
            sim_mc_std = 0.5

        guide_cfgs_uncond.append(GuideFnConfig(dist_pair=(x0, x1), guide_type='g_cov_A', guide_scale=0.0, ep=ep, energy_fn=energy_fn, energy_kwargs=energy_kwargs, cfm=cfm, ode_cfg=ODEConfig(t_end=t_end, num_steps=200, device=device)))
        guide_cfgs_ceg.append(GuideFnConfig(dist_pair=(x0, x1), guide_type='ceg', ep=ep, energy_fn=energy_fn, energy_kwargs=energy_kwargs, cfm=cfm, ode_cfg=ODEConfig(t_end=ceg_t_end, num_steps=200, device=device)))
        guide_cfgs_sim_mc.append(GuideFnConfig(dist_pair=(x0, x1), guide_type='g_sim_MC', sim_mc_std=sim_mc_std, sim_mc_n=200, ep=1e-2, guide_scale=sim_mc_scale, guide_schedule='linear_decay', energy_fn=energy_fn, energy_kwargs=energy_kwargs, cfm=cfm, ode_cfg=ODEConfig(t_end=t_end, num_steps=200, device=device)))
        guide_cfgs_g_cov_a.append(GuideFnConfig(dist_pair=(x0, x1), guide_type='g_cov_A', guide_scale=g_scale, guide_schedule=g_schedule, ep=ep, energy_fn=energy_fn, energy_kwargs=energy_kwargs, cfm=cfm, ode_cfg=ODEConfig(t_end=t_end, num_steps=200, device=device)))
        guide_cfgs_g_cov_g.append(GuideFnConfig(dist_pair=(x0, x1), guide_type='g_cov_G', guide_scale=g_scale, guide_schedule=g_schedule, ep=ep, energy_fn=energy_fn, energy_kwargs=energy_kwargs, cfm=cfm, ode_cfg=ODEConfig(t_end=t_end, num_steps=200, device=device)))
        guide_cfgs_mc.append(GuideFnConfig(dist_pair=(x0, x1), mc_batch_size=mc_batch_size, ep=mc_ep, scale=1, energy_fn=energy_fn, energy_kwargs=energy_kwargs, cfm=cfm, ode_cfg=ODEConfig(t_end=mc_t_end, num_steps=200, device=device)))
        guide_cfgs_gm.append(GuideFnConfig(dist_pair=(x0, x1), guide_type='learned', ep=ep, gm_type='g3', energy_fn=energy_fn, energy_kwargs=energy_kwargs, cfm=cfm, ode_cfg=ODEConfig(t_end=t_end, num_steps=200, device=device)))

    print("Evaluating Unconditional CFM...")
    trajs_uncond, _ = sample_and_compute_w2(guide_cfgs_uncond)
    print("Evaluating CEG...")
    trajs_ceg, _ = sample_and_compute_w2(guide_cfgs_ceg, show_plots=show_energy_plots)
    print("Evaluating Sim-MC...")
    trajs_sim_mc, _ = sample_and_compute_w2(guide_cfgs_sim_mc)
    print("Evaluating Guidance A...")
    trajs_g_cov_a, _ = sample_and_compute_w2(guide_cfgs_g_cov_a)
    print("Evaluating Guidance G...")
    trajs_g_cov_g, _ = sample_and_compute_w2(guide_cfgs_g_cov_g)
    print("Evaluating MC-CFM...")
    trajs_mc, _ = sample_and_compute_w2(guide_cfgs_mc)
    print("Evaluating GM-CFM...")
    trajs_gm, _ = sample_and_compute_w2(guide_cfgs_gm)
    
    trajs_cc = []
    cfgs_cc = []
    
    for i, pair_info in enumerate(dist_pairs):
        x0, x1 = pair_info[0], pair_info[1]
        # We only compute class-conditional CFM if target has canonical classes
        if x1 in ['8gaussian', 'moon']:
            num_classes = 8
            if cfm == "ot_cfm":
                folder_name = f"ot_cc_cfm_{x0}_{x1}"
            elif cfm == "entropic_ot_cfm":
                folder_name = f"entropic_ot_cc_cfm_{x0}_{x1}"
            else:
                folder_name = f"cc_cfm_{x0}_{x1}"
            log_dir = str(Path.cwd().parent / "logs" / f"{x0}-{x1}" / folder_name)
            cc_prefix = folder_name
            try:
                latest_cc_path = get_latest_model(log_dir, cc_prefix)
                print(f"Loading Class-conditional CFM model from: {latest_cc_path}")
                
                cfg_mc = guide_cfgs_mc[i]
                
                # Evaluate empirical class probabilities
                x1_dist = get_distribution(x1, energy_fn=cfg_mc.energy_fn, **cfg_mc.energy_kwargs)
                gt_x1 = sample_x1_frompeJ(x1_dist.sample, x1_dist, device, B=50000)
                angles = torch.atan2(gt_x1[:, 1], gt_x1[:, 0])
                gt_y = torch.round((angles + torch.pi) / (torch.pi / 4)) % 8
                class_counts = torch.bincount(gt_y.long().to(device), minlength=8)
                class_probs = class_counts.float() / class_counts.sum()
                
                cc_model = ClassConditionalMLP(
                    dim=2, 
                    num_classes=8, 
                    class_emb_dim=16, 
                    w=MLP_WIDTH_CCFM, 
                    time_varying=True, 
                    time_embedding=True
                ).to(device)
                cc_model.load_state_dict(torch.load(latest_cc_path, map_location=device))
                cc_model.eval()
                
                print(f"Sampling CC-CFM Baseline for {x0} -> {x1}...")
                traj_cc_cfm = sample_cc_cfm(cc_model, cfg_mc, class_probs=class_probs)
                trajs_cc.append(traj_cc_cfm)
                cfgs_cc.append(cfg_mc)
            except FileNotFoundError:
                print(f"No CC-CFM Baseline found for {x0} -> {x1}. Skipping...")
                trajs_cc.append(None)
                cfgs_cc.append(None)
        else:
            print(f"Target distribution '{x1}' has no canonical class division. Skipping CC-CFM.")
            trajs_cc.append(None)
            cfgs_cc.append(None)
            
    flat_trajs = []
    flat_cfgs = []
    all_results = {}
    
    for i, pair_info in enumerate(dist_pairs):
        x0, x1 = pair_info[0], pair_info[1]
        row_trajs = [
            trajs_uncond[i],
            trajs_ceg[i],
            trajs_sim_mc[i],
            trajs_g_cov_a[i],
            trajs_g_cov_g[i],
            trajs_mc[i],
            trajs_gm[i],
            trajs_cc[i]
        ]
        row_cfgs = [
            guide_cfgs_uncond[i],
            guide_cfgs_ceg[i],
            guide_cfgs_sim_mc[i],
            guide_cfgs_g_cov_a[i],
            guide_cfgs_g_cov_g[i],
            guide_cfgs_mc[i],
            guide_cfgs_gm[i],
            cfgs_cc[i]
        ]
        flat_trajs.extend(row_trajs)
        flat_cfgs.extend(row_cfgs)
        
        all_results[(x0, x1)] = {
            'uncond': (trajs_uncond[i], guide_cfgs_uncond[i]),
            'ceg': (trajs_ceg[i], guide_cfgs_ceg[i]),
            'sim_mc': (trajs_sim_mc[i], guide_cfgs_sim_mc[i]),
            'g_cov_a': (trajs_g_cov_a[i], guide_cfgs_g_cov_a[i]),
            'g_cov_g': (trajs_g_cov_g[i], guide_cfgs_g_cov_g[i]),
            'mc': (trajs_mc[i], guide_cfgs_mc[i]),
            'gm': (trajs_gm[i], guide_cfgs_gm[i]),
            'cc': (trajs_cc[i], cfgs_cc[i])
        }
        
    fig = plot_traj(flat_trajs, flat_cfgs, disp_b=256, nmethods=8)
    return fig, all_results


In [3]:
def sample_cc_cfm(model, cfg, class_probs=None):
    """
    Samples from the Class-Conditional Flow Matching model.
    
    Args:
        model: The trained ClassConditionalMLP.
        cfg: The GuideFnConfig containing dataset pairs and ODE parameters.
        class_probs: Optional tensor of probabilities for each of the 8 classes. 
                     If None, it generates classes uniformly (12.5% each).
    """
    device = cfg.ode_cfg.device
    B = cfg.ode_cfg.batch_size
    
    # 1. Handle Class Distributions
    if class_probs is not None:
        # Sample classes based on the provided empirical distribution
        y_target = torch.multinomial(class_probs, B, replacement=True).to(device)
    else:
        # Fallback to uniform distribution across the 8 classes
        y_target = torch.arange(8).repeat_interleave(B // 8).to(device)
        # Pad if batch size is not perfectly divisible by 8
        if len(y_target) < B:
            pad = torch.zeros(B - len(y_target), dtype=torch.long, device=device)
            y_target = torch.cat([y_target, pad])
            
    # 2. Setup ODE Wrapper
    wrapper = ClassConditionalMLPWrapper(model, y=y_target)
    node = NeuralODE(wrapper, solver="euler", sensitivity="adjoint")
    
    # 3. Sample from source distribution
    x0_dist = get_distribution(cfg.dist_pair[0]) 
    x0 = x0_dist.sample(B).to(device)
    
    # 4. Integrate
    with torch.no_grad():
        traj = node.trajectory(
            x0, 
            t_span=torch.linspace(0, cfg.ode_cfg.t_end, cfg.ode_cfg.num_steps, dtype=torch.float32).to(device)
        )
        
    return traj

In [4]:
def plot_traj(trajs, cfgs: List[GuideFnConfig], disp_b, skip_ref=False, nmethods=8):
    """
    Plots the trajectories for the ref distributions and the different methods.
    
    Args:
        trajs: Flat list of trajectories (tensors) across all rows and methods.
        cfgs: Flat list of GuideFnConfig configurations corresponding to each trajectory.
        disp_b: Number of trajectories to display on the plot.
        skip_ref: If True, skips plotting the reference distributions in the first two columns.
        nmethods: Number of baseline/comparison columns to plot.
        
    Returns:
        fig: The plotted Figure object.
    """
    blue = plt.get_cmap('coolwarm', 2)(0)
    red = plt.get_cmap('coolwarm', 2)(1)
    
    ndata = len(cfgs) // nmethods
    ncols = nmethods + 2
    
    # visualize, no VF
    fig, axs = plt.subplots(ndata, ncols, squeeze=False) 
    r = 6
    # Scale figure size dynamically based on number of columns
    fig.set_size_inches(ncols * r, ndata * r)
    
    size_ratio = 0.7
    
    # 1. plot ref distributions
    for i in range(ndata):
        if skip_ref:
            break
        # Always grab the 0th configuration of the row to establish limits
        row_cfg = cfgs[i * nmethods]
        
        x0_dist = get_distribution(row_cfg.dist_pair[0])
        x1_dist = get_distribution(row_cfg.dist_pair[1], energy_fn=row_cfg.energy_fn, **row_cfg.energy_kwargs)
        x0_sampler = x0_dist.sample
        x1_sampler = x1_dist.sample
        x1 = x1_sampler(1024).to(row_cfg.ode_cfg.device)
        
        ax = axs[i, 0]
        ax.scatter(x1[:, 0].cpu(), x1[:, 1].cpu(), s=3, color=red)
        ax.set_xlim(-row_cfg.xlim * size_ratio, row_cfg.xlim * size_ratio)
        ax.set_ylim(-row_cfg.ylim * size_ratio, row_cfg.ylim * size_ratio)
        
        # Hide ticks and spines, but keep ylabel visible
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
            
        def format_name(name):
            mapping = {
                'uniform': 'Uniform',
                '8gaussian': '8-Gaussian',
                'circle': 'Circle',
                's_curve': 'S-Curve',
                'moon': 'Moon'
            }
            return mapping.get(name, name.capitalize())
            
        row_name = f"{format_name(row_cfg.dist_pair[0])}\n$\\rightarrow$ {format_name(row_cfg.dist_pair[1])}"
        ax.set_ylabel(row_name, fontsize=32, fontweight='bold', labelpad=20)
        ax.set_aspect('auto')

        ax = axs[i, 1]
        # use rejection sampling to sample with weight e^-J in x1
        x1 = sample_x1_frompeJ(x1_sampler, x1_dist, row_cfg.ode_cfg.device, 1024)
        ax.scatter(x1[:, 0].cpu(), x1[:, 1].cpu(), s=3, color=red)
        ax.set_xlim(-row_cfg.xlim * size_ratio, row_cfg.xlim * size_ratio)
        ax.set_ylim(-row_cfg.ylim * size_ratio, row_cfg.ylim * size_ratio)
        ax.axis('off')
        ax.set_aspect('auto')


    # 2. plot different methods
    for i in range(ndata):
        # We need the row's base config to ensure limits are consistent across the whole row
        row_cfg = cfgs[i * nmethods] 
        
        for j in range(nmethods):
            ax = axs[i, j + 2]
            
            # Apply consistent limits to EVERY plot in the row, even empty ones
            ax.set_xlim(-row_cfg.xlim * size_ratio, row_cfg.xlim * size_ratio)
            ax.set_ylim(-row_cfg.ylim * size_ratio, row_cfg.ylim * size_ratio)
            ax.axis('off')
            ax.set_aspect('auto')
            
            try:
                traj = trajs[i * nmethods + j]
                cfg = cfgs[i * nmethods + j]
                
                # Safely skip the None placeholders we added, leaving a blank square
                if traj is None or cfg is None:
                    continue
            except IndexError:
                continue
            
            idx = torch.randperm(cfg.ode_cfg.batch_size)[:disp_b]
            
            # plot the start and end points with endpoint colors in the colormap
            ax.scatter(traj[0, :, 0].cpu(), traj[0, :, 1].cpu(), s=3, color=blue)
            ax.scatter(traj[-1, :, 0].cpu(), traj[-1, :, 1].cpu(), s=3, color=red)
            
            # plot the trajectory with a gradient color
            colors = torch.linspace(0, 1, cfg.ode_cfg.num_steps).unsqueeze(1).repeat(1, disp_b).flatten().numpy()
            ax.scatter(
                traj[:, idx, 0].flatten().cpu(), 
                traj[:, idx, 1].flatten().cpu(), 
                c=plt.cm.coolwarm(colors), 
                alpha=0.5, s=0.1, marker='.'
            )
            
    # Add column titles on the first row
    titles = [
        r"$\bf{Target}$" + "\n" + r"$p(x_1)$",
        r"$\bf{Ground\ Truth}$" + "\n" + r"$p(x_1) e^{-J(x_1)} / Z$",
        r"$\bf{CFM}$" + "\n" + r"$p(x_1)$",
        r"$\bf{CEG}$",
        r"$g^{\bf{sim-MC}}$",
        r"$g^{\bf{cov-A}}$",
        r"$g^{\bf{cov-G}}$",
        r"$g^{\bf{MC}}$",
        r"$g_{\phi}$",
        r"$\bf{CCFM}$"
    ]
    if ndata > 0:
        for col_idx in range(ncols):
            if col_idx < len(titles):
                axs[0, col_idx].set_title(titles[col_idx], fontsize=32)
                
    return fig


In [ ]:
dist_pairs = [
    # ('circle', 's_curve'),
    ('uniform', '8gaussian', 'single_blob_quadratic_distance', {'target_blob_idx': 0, 'std_target': 0.1}),
    # ('uniform', '8gaussian', 'three_blobs_continuous_multi_well'),  # default 0, 2, 5
    # ('8gaussian', 'moon', 'upper_moon_classifier_score'),
]
fig, all_results = run_and_plot_comparisons(dist_pairs)
fig.savefig('toy_with_cc_baseline_fixed.png', dpi=100, bbox_inches='tight', pad_inches=0)


Using device: mps
Evaluating Unconditional CFM...
Monte Carlo batch size: 1024
Loaded latest model: cfm_uniform_8gaussian_20260608_180938.pth
Evaluating CEG...
Monte Carlo batch size: 1024
Loaded latest model: cfm_uniform_8gaussian_20260608_180938.pth
[ Verification ] Loaded training config from ../logs/uniform-8gaussian/cfm_uniform_8gaussian/ceg_scale_1.0_uniform_8gaussian_20260612_121912.yaml


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /Users/LS/.netrc.


[ Verification ] Found corresponding WandB run: ceg_scale_1.0_8gaussian_MoonDistribution_20260612_121918
[ Verification ] Energy function and hyperparameter compatibility successfully verified!
Loaded latest compatible model: ceg_scale_1.0_uniform_8gaussian_20260612_121912.pth
Evaluating Sim-MC...
Monte Carlo batch size: 1024
Loaded latest model: cfm_uniform_8gaussian_20260608_180938.pth
Evaluating Guidance A...
Monte Carlo batch size: 1024
Loaded latest model: cfm_uniform_8gaussian_20260608_180938.pth
Evaluating Guidance G...
Monte Carlo batch size: 1024
Loaded latest model: cfm_uniform_8gaussian_20260608_180938.pth
Evaluating MC-CFM...
Monte Carlo batch size: 10240
Loaded latest model: cfm_uniform_8gaussian_20260608_180938.pth


In [ ]:
def compute_and_display_w2_tables(all_results, dist_pairs, eval_b=2048):
    import pandas as pd
    import torch
    from pathlib import Path
    
    for pair_info in dist_pairs:
        if len(pair_info) == 2:
            x0, x1 = pair_info
        elif len(pair_info) == 3:
            x0, x1, _ = pair_info
        else:
            x0, x1, _, _ = pair_info
            
        print(f"\n============================================================")
        print(f"Evaluating W2 Table for {x0} -> {x1} (B={eval_b})")
        print(f"============================================================")
        
        results_pair = all_results.get((x0, x1))
        if results_pair is None:
            print(f"No results found for {x0} -> {x1}. Skipping...")
            continue
            
        # 1. Update the batch size in configs and re-sample
        generated = {}
        device = results_pair['mc'][1].ode_cfg.device
        
        # Baselines mapping
        baselines = {
            "CFM": 'uncond',
            "CEG": 'ceg',
            "g_sim_MC": 'sim_mc',
            "g_cov_A": 'g_cov_a',
            "g_cov_G": 'g_cov_g',
            "MC": 'mc',
            "Learned (GM)": 'gm'
        }
        
        for name, key in baselines.items():
            traj_data, cfg = results_pair[key]
            # Set batch size to eval_b
            cfg.ode_cfg.batch_size = eval_b
            print(f"Sampling {name} for {x0} -> {x1}...")
            # Re-sample
            trajs, _ = sample_and_compute_w2([cfg])
            generated[name] = trajs[0][-1]
            
        # 2. Class-conditional CFM (if applicable)
        if x1 in ['8gaussian', 'moon']:
            cfm_type = results_pair['mc'][1].cfm
            if cfm_type == "ot_cfm":
                folder_name = f"ot_cc_cfm_{x0}_{x1}"
            elif cfm_type == "entropic_ot_cfm":
                folder_name = f"entropic_ot_cc_cfm_{x0}_{x1}"
            else:
                folder_name = f"cc_cfm_{x0}_{x1}"
            log_dir = str(Path.cwd().parent / "logs" / f"{x0}-{x1}" / folder_name)
            cc_prefix = folder_name
            try:
                latest_cc_path = get_latest_model(log_dir, cc_prefix)
                print(f"Loading Class-conditional CFM model from: {latest_cc_path}")
                
                cfg_mc = results_pair['mc'][1]
                # Update batch size
                cfg_mc.ode_cfg.batch_size = eval_b
                
                # Evaluate empirical class probabilities
                x1_dist = get_distribution(x1, energy_fn=cfg_mc.energy_fn, **cfg_mc.energy_kwargs)
                gt_x1 = sample_x1_frompeJ(x1_dist.sample, x1_dist, device, B=50000)
                angles = torch.atan2(gt_x1[:, 1], gt_x1[:, 0])
                gt_y = torch.round((angles + torch.pi) / (torch.pi / 4)) % 8
                class_counts = torch.bincount(gt_y.long().to(device), minlength=8)
                class_probs = class_counts.float() / class_counts.sum()
                
                cc_model = ClassConditionalMLP(
                    dim=2, 
                    num_classes=8, 
                    class_emb_dim=16, 
                    w=MLP_WIDTH_CCFM, 
                    time_varying=True, 
                    time_embedding=True
                ).to(device)
                cc_model.load_state_dict(torch.load(latest_cc_path, map_location=device))
                cc_model.eval()
                
                print(f"Sampling Class-cond. CFM for {x0} -> {x1}...")
                traj_cc_large = sample_cc_cfm(cc_model, cfg_mc, class_probs=class_probs)
                generated["Class-cond. CFM"] = traj_cc_large[-1]
            except FileNotFoundError:
                print(f"No CC-CFM Baseline found for {x0} -> {x1}. Skipping CC-CFM in table.")
        
        # 3. Sample True target distribution
        cfg_mc = results_pair['mc'][1]
        x1_dist = get_distribution(x1, energy_fn=cfg_mc.energy_fn, **cfg_mc.energy_kwargs)
        x1_true = sample_x1_frompeJ(
            x1_dist.sample,
            x1_dist,
            device,
            B=eval_b,
        )
        
        # 4. Compute W2, KL, and Rev KL metrics
        results = []
        with torch.no_grad():
            for name, x_gen in generated.items():
                w2_score = w2(
                    x_gen.detach()[:eval_b],
                    x1_true.detach()[:eval_b],
                )
                if torch.is_tensor(w2_score):
                    w2_score = w2_score.item()
                
                # Compute KL and Reverse KL
                kl_val, rev_kl_val = compute_kl_divergence(
                    x1_dist, 
                    x_gen.detach()[:eval_b], 
                    scale=cfg_mc.scale
                )
                results.append({
                    "Method": name,
                    "W2 ↓": w2_score,
                    "KL ↓": kl_val,
                    "Rev KL ↓": rev_kl_val
                })
                
        df_metrics = (
            pd.DataFrame(results)
            .sort_values("W2 ↓")
            .reset_index(drop=True)
        )
        df_metrics["W2 ↓"] = df_metrics["W2 ↓"].map("{:.4f}".format)
        df_metrics["KL ↓"] = df_metrics["KL ↓"].map("{:.4f}".format)
        df_metrics["Rev KL ↓"] = df_metrics["Rev KL ↓"].map("{:.4f}".format)
        try:
            display(df_metrics)
        except NameError:
            print(df_metrics)


In [ ]:
# ============================================================
# Final evaluation:
# Wasserstein distance
# between generated distribution and p*(x) ∝ p(x) exp(-J(x))
# ============================================================

compute_and_display_w2_tables(all_results, dist_pairs, eval_b=2048)

In [ ]:
def compute_and_display_combined_w2_table(comparison_results, dist_pairs, eval_b=2048):
    import pandas as pd
    import torch
    from pathlib import Path
    
    for pair_info in dist_pairs:
        if len(pair_info) == 2:
            x0, x1 = pair_info
        elif len(pair_info) == 3:
            x0, x1, _ = pair_info
        else:
            x0, x1, _, _ = pair_info
            
        print(f"\n============================================================")
        print(f"Evaluating Combined W2 Table for {x0} -> {x1} (B={eval_b})")
        print(f"============================================================")
        
        # Baselines mapping
        baselines = {
            "CFM": 'uncond',
            "CEG": 'ceg',
            "g_sim_MC": 'sim_mc',
            "g_cov_A": 'g_cov_a',
            "g_cov_G": 'g_cov_g',
            "MC": 'mc',
            "Learned (GM)": 'gm'
        }
        
        # We will collect W2 results for each coupling method
        w2_results = {
            "Method": list(baselines.keys()) + (["Class-cond. CFM"] if x1 in ['8gaussian', 'moon'] else [])
        }
        
        # We'll initialize columns for cfm, ot_cfm, and entropic_ot_cfm
        for cfm_name in ['cfm', 'ot_cfm', 'entropic_ot_cfm']:
            col_name = "W2 of cfm" if cfm_name == 'cfm' else ("W2 of ot_cfm" if cfm_name == 'ot_cfm' else "entropic_ot_cfm")
            w2_results[col_name] = []
            
            results_pair = comparison_results.get(cfm_name, {}).get((x0, x1))
            if results_pair is None:
                print(f"No results found for {cfm_name} on {x0} -> {x1}.")
                w2_results[col_name] = [float('nan')] * len(w2_results["Method"])
                continue
                
            device = results_pair['mc'][1].ode_cfg.device
            cfg_mc = results_pair['mc'][1]
            x1_dist = get_distribution(x1, energy_fn=cfg_mc.energy_fn, **cfg_mc.energy_kwargs)
            x1_true = sample_x1_frompeJ(x1_dist.sample, x1_dist, device, B=eval_b)
            
            # 1. Sample standard baselines
            generated = {}
            for name, key in baselines.items():
                traj_data, cfg = results_pair[key]
                cfg.ode_cfg.batch_size = eval_b
                trajs, _ = sample_and_compute_w2([cfg])
                generated[name] = trajs[0][-1]
                
            # 2. Sample class-conditional baseline
            if x1 in ['8gaussian', 'moon']:
                if cfm_name == "ot_cfm":
                    folder_name = f"ot_cc_cfm_{x0}_{x1}"
                elif cfm_name == "entropic_ot_cfm":
                    folder_name = f"entropic_ot_cc_cfm_{x0}_{x1}"
                else:
                    folder_name = f"cc_cfm_{x0}_{x1}"
                log_dir = str(Path.cwd().parent / "logs" / f"{x0}-{x1}" / folder_name)
                cc_prefix = folder_name
                try:
                    latest_cc_path = get_latest_model(log_dir, cc_prefix)
                    cfg_mc.ode_cfg.batch_size = eval_b
                    gt_x1 = sample_x1_frompeJ(x1_dist.sample, x1_dist, device, B=50000)
                    angles = torch.atan2(gt_x1[:, 1], gt_x1[:, 0])
                    gt_y = torch.round((angles + torch.pi) / (torch.pi / 4)) % 8
                    class_counts = torch.bincount(gt_y.long().to(device), minlength=8)
                    class_probs = class_counts.float() / class_counts.sum()
                    
                    cc_model = ClassConditionalMLP(
                        dim=2, 
                        num_classes=8, 
                        class_emb_dim=16, 
                        w=MLP_WIDTH_CCFM, 
                        time_varying=True, 
                        time_embedding=True
                    ).to(device)
                    cc_model.load_state_dict(torch.load(latest_cc_path, map_location=device))
                    cc_model.eval()
                    
                    traj_cc_large = sample_cc_cfm(cc_model, cfg_mc, class_probs=class_probs)
                    generated["Class-cond. CFM"] = traj_cc_large[-1]
                except FileNotFoundError:
                    generated["Class-cond. CFM"] = None
                    
            # 3. Compute W2 scores
            for name in w2_results["Method"]:
                x_gen = generated.get(name)
                if x_gen is not None:
                    score = w2(x_gen.detach()[:eval_b], x1_true.detach()[:eval_b])
                    if torch.is_tensor(score):
                        score = score.item()
                    w2_results[col_name].append(score)
                else:
                    w2_results[col_name].append(float('nan'))
                    
        # Create DataFrame
        df_combined = pd.DataFrame(w2_results)
        # Format the W2 columns to 4 decimal places
        for cfm_name in ['cfm', 'ot_cfm', 'entropic_ot_cfm']:
            col_name = "W2 of cfm" if cfm_name == 'cfm' else ("W2 of ot_cfm" if cfm_name == 'ot_cfm' else "entropic_ot_cfm")
            df_combined[col_name] = df_combined[col_name].map(lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A")
            
        try:
            display(df_combined)
        except NameError:
            print(df_combined)


In [ ]:
dist_pairs_single_blob = [
    ('uniform', '8gaussian', 'single_blob_quadratic_distance', {'target_blob_idx': 0, 'std_target': 0.1}),
]

comparison_results = {}
for cfm_name in ['cfm', 'ot_cfm', 'entropic_ot_cfm']:
    print(f"\n=============================================================")
    print(f" EVALUATING COUPLING: {cfm_name}")
    print(f"=============================================================")
    try:
        # Run evaluations and plot trajectories
        fig, all_results = run_and_plot_comparisons(dist_pairs_single_blob, cfm=cfm_name)
        fig.suptitle(f"Coupling Method: {cfm_name}", fontsize=36, y=1.02)
        fig.savefig(f'toy_single_blob_comparison_{cfm_name}.png', dpi=100, bbox_inches='tight', pad_inches=0.1)
        import matplotlib.pyplot as plt
        plt.show()
        
        comparison_results[cfm_name] = all_results
    except Exception as e:
        print(f"Error evaluating {cfm_name}: {e}")

# Compute and display the combined W2 table
compute_and_display_combined_w2_table(comparison_results, dist_pairs_single_blob, eval_b=2048)
